In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [4]:
# urls
base_article_url = "https://muse.jhu.edu"
name_index_url = 'https://muse.jhu.edu/ushmm/index/names'
vol_urls = ["https://muse.jhu.edu/resource_group/13", "2", "3", "4"]

# create session to save cookies
session = requests.Session()

In [20]:
# get urls for all of the content articles in a volume
# def scrape_vol(vol_url, plc_df, session, vol_num):
vol_response = session.get(vol_urls[0])
vol_response.raise_for_status()
vol_soup = BeautifulSoup(vol_response.text, 'lxml')
#     print("SOUP:", vol_soup)
#     print("\n\n\n")
# article_links = get_vol_article_links(vol_soup, dummy_plc_df, vol_num)
#     return article_links
vol_soup

<!DOCTYPE html>
<html lang="en">
<head>
<!-- Global site tag (gtag.js) - Google Analytics -->
<script async="" src="https://www.googletagmanager.com/gtag/js?id=UA-58347753-2"></script>
<script>
		  window.dataLayer = window.dataLayer || [];
		  function gtag(){dataLayer.push(arguments);}
		  gtag('js', new Date());		
		  gtag('config', 'UA-58347753-2');
		  gtag('config', 'G-9E7VT47G3J');
		</script>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<meta content="telephone=no" name="format-detection"/>
<meta content="/resource_group/13/og_image.jpg" property="og:image"/>
<!-- google -->
<!-- /google -->
<link href="/favicon.ico" rel="icon"/>
<title>Project MUSE - Volume I: Early Camps, Youth Camps, and Concentration Camps and Subcamps under the SS-Business Administration Main Office (WVHA)</title>
<style>
.js_on { display:none; }
</style>
<link href="/css/normalize.css" rel="stylesheet" type="text/css"/>
<link href="/css/foundation.min.css

In [21]:
# load in plc_df
plc_df = pd.read_csv("./../ecg_interpreter/tables/place_table.csv")
plc_df

,LID,Volume,Name,Type,Subtype,Current Country,Latitude,Longitude,Location Accuracy
0,0,1,'s-Gravenhage,Concentration Camp,Subcamp,Netherlands,52.0833,4.3000,Exact
1,1,1,Abteroda,Concentration Camp,Subcamp,Germany,50.9000,10.0667,Exact
2,2,1,Adlershorst,Concentration Camp,Subcamp,Poland,53.4500,20.5333,Exact
3,3,1,Ahrensbök-Holstendorf,Early Concentration Camp,NaN,Germany,54.0167,10.5833,Exact
4,4,1,"Alderney (Kanalinsel, SS-Bbde I)",Concentration Camp,Subcamp,United Kingdom,49.7060,-2.2145,Exact
...,...,...,...,...,...,...,...,...,...
3458,3458,4,WEHRMACHTGEFÄNGNIS FREIBURG,Wehrmacht Disciplinary Facilities,WG (Wehrmachtgefängnis),France,47.9868,7.5162,Exact
3459,3459,4,WEHRMACHTGEFÄNGNIS GERMERSHEIM,Wehrmacht Disciplinary Facilities,WG (Wehrmachtgefängnis),Germany,49.2170,8.3670,Exact
3460,3460,4,WEHRMACHTGEFÄNGNIS GLATZ,Wehrmacht Disciplinary Facilities,WG (Wehrmachtgefängnis),Poland,50.4330,16.6500,Exact
3461,3461,4,WEHRMACHTGEFÄNGNIS GRAUDENZ,Wehrmacht Disciplinary Facilities,WG (Wehrmachtgefängnis),Poland,53.4949,18.7529,Exact


In [52]:
# helper function - get dict of lid and article url (NOT including base url)
def get_vol_article_links(vol_soup, plc_df, vol_num):
    filtered_plc_df = plc_df[plc_df["Volume"] == vol_num]
#     print(filtered_plc_df)
    articles = {}
    article_cards = vol_soup.find_all(class_="card_text")
#     print(article_cards)
    # look at each article entry on the page
    # TEST: only get the first 5 content articles
    i = 0
    for card in article_cards:
        if i >= 5:
            return articles
#         print(card)
        title = card.find(class_="title").text
        # check if place name is in plc_df
        row = filtered_plc_df[filtered_plc_df["Name"].str.lower() == title.lower()]
        row = row.reset_index(drop=True)
        # case 1: article is a place
        if len(row) == 1:
            print("FOUND", title, "IN PLC_DF")
            doc_num = card.find(class_="action_btns")
#             print("\n\nACTION BUTTONS:\n", doc_num)
            entries = card.select("ol li a")
#             print("\n\nENTRIES:\n", entries)
            # document url is always the first list item
            if entries:
                # save lid and url ending into dict
                articles[int(row["LID"][0])] = entries[0]["href"]
                i += 1
        # case 2: article is not a place / other error
        else:
            print("NUMBER OF ROWS FOUND WITH", title, ":", len(row))
    return articles

In [54]:
# test volume 1 for the first 5 articles
article_links = get_vol_article_links(vol_soup, plc_df, 1)
# article_links = scrape_vol(vol_urls[0], plc_df, session, 1)
article_links

NUMBER OF ROWS FOUND WITH Volume I: Early Camps, Youth Camps, and Concentration Camps and Subcamps under the SS-Business Administration Main Office (WVHA) : 0
NUMBER OF ROWS FOUND WITH FRONTMATTER : 0
NUMBER OF ROWS FOUND WITH LIST OF MAPS : 0
NUMBER OF ROWS FOUND WITH FOREWORD : 0
NUMBER OF ROWS FOUND WITH PREFACE : 0
NUMBER OF ROWS FOUND WITH ACKNOWLEDGMENTS : 0
NUMBER OF ROWS FOUND WITH EDITOR’S INTRODUCTION TO THE SERIES AND VOLUME I : 0
NUMBER OF ROWS FOUND WITH READER’S GUIDE TO USING THE ENCYCLOPEDIA : 0
NUMBER OF ROWS FOUND WITH A NOTE ON THE RECENTLY OPENED INTERNATIONAL TRACING SERVICE DOCUMENTATION : 0
NUMBER OF ROWS FOUND WITH LIST OF ABBREVIATIONS : 0
NUMBER OF ROWS FOUND WITH LIST OF CONTRIBUTORS : 0
NUMBER OF ROWS FOUND WITH ABOUT THE EDITOR : 0
NUMBER OF ROWS FOUND WITH INTRODUCTION TO THE EARLY CAMPS : 0
FOUND AHRENSBÖK-HOLSTENDORF IN PLC_DF
FOUND ALT DABER IN PLC_DF
FOUND ALTENBERG IN PLC_DF
FOUND ANKENBUCK IN PLC_DF
FOUND ANRATH BEI KREFELD IN PLC_DF


{3: '/document/1261',
 6: '/document/1262',
 8: '/document/1263',
 14: '/document/1264',
 16: '/document/1265'}